In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

ledger   = pd.read_csv(BASE_PATH + 'stock_ledger.csv')
inv      = pd.read_csv(BASE_PATH + 'inventory_master.csv')
po_h     = pd.read_csv(BASE_PATH + 'purchase_orders_header.csv')
po_l     = pd.read_csv(BASE_PATH + 'purchase_orders_lines.csv')
so_h     = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
so_l     = pd.read_csv(BASE_PATH + 'sales_orders_lines.csv')
invoices = pd.read_csv(BASE_PATH + 'invoices.csv')
payments = pd.read_csv(BASE_PATH + 'payments.csv')
products = pd.read_csv(BASE_PATH + 'products.csv')
branches = pd.read_csv(BASE_PATH + 'branches.csv')

loaded = {'ledger': ledger, 'inv': inv, 'po_h': po_h, 'po_l': po_l, 'so_h': so_h,
          'so_l': so_l, 'invoices': invoices, 'payments': payments,
          'products': products, 'branches': branches}

for name, table in loaded.items():
    print(name, table.shape)

print()
print('invoices columns:', list(invoices.columns))
print('payments columns:', list(payments.columns))
print('branches columns:', list(branches.columns))
print()
print(invoices.head(3))

Mounted at /content/drive
ledger (237230, 9)
inv (180, 8)
po_h (24000, 10)
po_l (155495, 9)
so_h (20000, 11)
so_l (130402, 9)
invoices (18033, 10)
payments (19257, 5)
products (30, 23)
branches (6, 13)

invoices columns: ['invoice_id', 'so_id', 'customer_id', 'branch_id', 'invoice_date', 'due_date', 'total_order_value', 'total_gst_amount', 'grand_total', 'payment_status']
payments columns: ['payment_id', 'invoice_id', 'payment_date', 'payment_amount', 'payment_method']
branches columns: ['branch_id', 'branch_name', 'city', 'state', 'region', 'warehouse_type', 'warehouse_capacity', 'service_center_available', 'manager_id', 'total_employees', 'avg_monthly_revenue', 'monthly_operational_cost', 'market_demand_index']

   invoice_id      so_id customer_id branch_id invoice_date    due_date  \
0  INV-325361  SO-990591       C0070    AHM001   2022-05-28  2022-06-27   
1  INV-568476  SO-431510       C0031    KOL001   2022-05-08  2022-06-07   
2  INV-786913  SO-495859       C0013    AHM001   20

In [2]:
invoice_counts = invoices['invoice_id'].value_counts()
payment_counts = payments['payment_id'].value_counts()

clean_invoices = invoices[invoices['invoice_id'].map(invoice_counts) == 1].copy()
clean_payments = payments[payments['payment_id'].map(payment_counts) == 1].copy()

print('invoices left out for a repeated id:', len(invoices) - len(clean_invoices))
print('payments left out for a repeated id:', len(payments) - len(clean_payments))

clean_invoices['due_date'] = pd.to_datetime(clean_invoices['due_date'])
clean_payments['payment_date'] = pd.to_datetime(clean_payments['payment_date'])

paid = clean_payments.merge(
    clean_invoices[['invoice_id', 'branch_id', 'due_date']],
    on='invoice_id', how='inner')

paid['is_late'] = paid['payment_date'] > paid['due_date']

print('joined rows:', len(paid))
print('late payments:', int(paid['is_late'].sum()))
print('late rate overall %:', round(paid['is_late'].mean() * 100, 2))

late_by_branch = paid.groupby('branch_id')['is_late'].agg(
    payments='count', late='sum', late_rate_percent=lambda s: round(s.mean() * 100, 2))
print()
print(late_by_branch)
print()
print('branch rows add to:', int(late_by_branch['payments'].sum()), 'payments and',
      int(late_by_branch['late'].sum()), 'late')

invoices left out for a repeated id: 393
payments left out for a repeated id: 403
joined rows: 18445
late payments: 7655
late rate overall %: 41.5

           payments  late  late_rate_percent
branch_id                                   
AHM001         3196  1266              39.61
CHN001         3732  1814              48.61
DEL001         2723  1027              37.72
HYD001         2576   984              38.20
KOL001         3488  1386              39.74
PUN001         2730  1178              43.15

branch rows add to: 18445 payments and 7655 late


In [3]:
from scipy.stats import chi2_contingency

late_table = pd.crosstab(paid['branch_id'], paid['is_late'])
chi2, p_value, dof, expected = chi2_contingency(late_table)

print(late_table)
print()
print('p-value, late against branch:', round(p_value, 6))
print('smallest expected count in any cell:', round(expected.min(), 1))

is_late    False  True 
branch_id              
AHM001      1930   1266
CHN001      1918   1814
DEL001      1696   1027
HYD001      1592    984
KOL001      2102   1386
PUN001      1552   1178

p-value, late against branch: 0.0
smallest expected count in any cell: 1069.1


In [4]:
pair = ['product_id', 'branch_id']

led = ledger.copy()
led['movement_date'] = pd.to_datetime(led['movement_date'])
led = led.sort_values(pair + ['movement_date'], kind='mergesort').reset_index(drop=True)
led = led.merge(inv[pair + ['opening_stock']], on=pair, how='left')
led['previous_balance'] = led.groupby(pair)['running_balance'].shift(1)
led['previous_balance'] = led['previous_balance'].fillna(led['opening_stock'])
led['balance_change'] = led['running_balance'] - led['previous_balance']

net_adjustments = led[led['movement_type'] == 'ADJUSTMENT'].groupby(pair)['balance_change'].sum()

received_lines = po_l.merge(po_h[['po_id', 'branch_id', 'po_status']], on='po_id')
received = received_lines[received_lines['po_status'] == 'Received'].groupby(pair)['quantity'].sum()

delivered_lines = so_l.merge(so_h[['so_id', 'branch_id', 'order_status', 'delivery_date']], on='so_id')
delivered_lines = delivered_lines[delivered_lines['order_status'] == 'Delivered'].copy()
delivered_lines['delivery_date'] = pd.to_datetime(delivered_lines['delivery_date'])
delivered = delivered_lines.groupby(pair)['quantity'].sum()
sold_last_year = delivered_lines[delivered_lines['delivery_date'] > '2024-01-14'].groupby(pair)['quantity'].sum()

risk_stock = inv.set_index(pair)[['opening_stock']].copy()
risk_stock['received'] = received
risk_stock['delivered'] = delivered
risk_stock['net_adjustments'] = net_adjustments
risk_stock['units_sold'] = sold_last_year
risk_stock = risk_stock.fillna(0)
risk_stock['order_line_stock'] = (risk_stock['opening_stock'] + risk_stock['received']
                                  - risk_stock['delivered'] + risk_stock['net_adjustments'])
risk_stock['years_of_cover'] = risk_stock['order_line_stock'] / risk_stock['units_sold']

print('pairs:', len(risk_stock))
print('order-line stock, all pairs:', int(risk_stock['order_line_stock'].sum()))
print('net adjustments, all pairs:', int(risk_stock['net_adjustments'].sum()))
print('units sold in the window, all pairs:', int(risk_stock['units_sold'].sum()))
print()
print(risk_stock.loc[('P005', 'HYD001'), ['order_line_stock', 'units_sold', 'years_of_cover']].round(2))
print()

cover_by_branch = risk_stock.groupby('branch_id')[['order_line_stock', 'units_sold']].sum()
cover_by_branch['years_of_cover'] = (cover_by_branch['order_line_stock'] / cover_by_branch['units_sold']).round(2)
print(cover_by_branch)

pairs: 180
order-line stock, all pairs: 21185215
net adjustments, all pairs: 62
units sold in the window, all pairs: 197300

order_line_stock    122924.00
units_sold             715.00
years_of_cover         171.92
Name: (P005, HYD001), dtype: float64

           order_line_stock  units_sold  years_of_cover
branch_id                                              
AHM001            3472113.0       31376          110.66
CHN001            3548773.0       39930           88.87
DEL001            3597678.0       31565          113.98
HYD001            3555196.0       27484          129.36
KOL001            3435814.0       36543           94.02
PUN001            3575641.0       30402          117.61


In [5]:
from scipy.stats import kruskal

cover_groups = [group['years_of_cover'].values
                for branch, group in risk_stock.reset_index().groupby('branch_id')]
h_value, p_cover = kruskal(*cover_groups)

median_cover = risk_stock.reset_index().groupby('branch_id')['years_of_cover'].median().round(2)
print(median_cover)
print()
print('pairs per branch:', [len(g) for g in cover_groups])
print('p-value, years of cover against branch:', round(p_cover, 6))

branch_id
AHM001    113.19
CHN001     89.47
DEL001    117.92
HYD001    131.74
KOL001     92.13
PUN001    114.83
Name: years_of_cover, dtype: float64

pairs per branch: [30, 30, 30, 30, 30, 30]
p-value, years of cover against branch: 0.0


In [6]:
from scipy.stats import chisquare

moves = led[led['movement_type'].isin(['IN', 'OUT'])]
handled_units = moves.groupby('branch_id')['quantity'].sum()
handled_rows = moves.groupby('branch_id')['quantity'].count()

adjustments = led[led['movement_type'] == 'ADJUSTMENT']
subtracted_part = adjustments[adjustments['balance_change'] < 0]
subtracted_units = subtracted_part.groupby('branch_id')['quantity'].sum()
subtracted_rows = subtracted_part.groupby('branch_id')['quantity'].count()

loss_by_branch = pd.DataFrame({
    'handled_units': handled_units,
    'subtracted_units': subtracted_units,
    'handled_rows': handled_rows,
    'subtracted_rows': subtracted_rows})
loss_by_branch['loss_rate_percent'] = (loss_by_branch['subtracted_units'] / loss_by_branch['handled_units'] * 100).round(4)

print(loss_by_branch)
print()
print('units handled:', int(handled_units.sum()))
print('units subtracted:', int(subtracted_units.sum()))
print('loss rate overall %:', round(subtracted_units.sum() / handled_units.sum() * 100, 4))

expected_rows = subtracted_rows.sum() * handled_rows / handled_rows.sum()
chi_value, p_loss = chisquare(subtracted_rows, expected_rows)
print()
print('p-value, subtraction rows against handled rows by branch:', round(p_loss, 4))

           handled_units  subtracted_units  handled_rows  subtracted_rows  \
branch_id                                                                   
AHM001           3539666              5190         39525              196   
CHN001           3692804              5980         43600              226   
DEL001           3622464              5431         37575              202   
HYD001           3548234              5730         35877              207   
KOL001           3538563              6200         40848              218   
PUN001           3580896              4933         37351              178   

           loss_rate_percent  
branch_id                     
AHM001                0.1466  
CHN001                0.1619  
DEL001                0.1499  
HYD001                0.1615  
KOL001                0.1752  
PUN001                0.1378  

units handled: 21522627
units subtracted: 33464
loss rate overall %: 0.1555

p-value, subtraction rows against handled rows by branch:

In [7]:
def mark_side(movement, header, header_id, status_col, lines):
    rows = led[led['movement_type'] == movement][
        ['movement_id', 'product_id', 'branch_id', 'reference_id', 'quantity']].copy()
    rows = rows.merge(header[[header_id, status_col]],
                      left_on='reference_id', right_on=header_id, how='left')
    rows['cancelled_order'] = rows[status_col] == 'Cancelled'

    ledger_units = (rows.groupby(['reference_id', 'product_id'])['quantity']
                        .sum().rename('ledger_units').reset_index())
    line_units = (lines.groupby([header_id, 'product_id'])['quantity']
                       .sum().rename('line_units').reset_index())
    both = ledger_units.merge(line_units, left_on=['reference_id', 'product_id'],
                              right_on=[header_id, 'product_id'], how='left')
    both['lower'] = both['ledger_units'] < both['line_units']

    rows = rows.merge(both[['reference_id', 'product_id', 'lower']],
                      on=['reference_id', 'product_id'], how='left')
    rows['lower_than_order'] = rows['lower'].fillna(False).astype(bool) & ~rows['cancelled_order']
    return rows[['movement_id', 'branch_id', 'cancelled_order', 'lower_than_order']]

flagged_rows = pd.concat([
    mark_side('IN', po_h, 'po_id', 'po_status', po_l),
    mark_side('OUT', so_h, 'so_id', 'order_status', so_l)])
flagged_rows['flagged_any'] = flagged_rows['cancelled_order'] | flagged_rows['lower_than_order']

print('ledger rows:', len(led))
print('rows tied to cancelled orders:', int(flagged_rows['cancelled_order'].sum()))
print('rows below the order-line units:', int(flagged_rows['lower_than_order'].sum()))
print('rows flagged in either way:', int(flagged_rows['flagged_any'].sum()))
print('share of the ledger flagged %:', round(flagged_rows['flagged_any'].sum() / len(led) * 100, 2))

flag_by_branch = pd.DataFrame({
    'ledger_rows': led.groupby('branch_id').size(),
    'flagged_rows': flagged_rows.groupby('branch_id')['flagged_any'].sum()})
flag_by_branch['flagged_share_percent'] = (flag_by_branch['flagged_rows'] / flag_by_branch['ledger_rows'] * 100).round(2)
print()
print(flag_by_branch)

flag_table = pd.DataFrame({
    'not_flagged': flag_by_branch['ledger_rows'] - flag_by_branch['flagged_rows'],
    'flagged': flag_by_branch['flagged_rows']})
chi_flag, p_flag, dof_flag, expected_flag = chi2_contingency(flag_table)
print()
print('p-value, flagged against branch:', round(p_flag, 4))

ledger rows: 237230
rows tied to cancelled orders: 2817
rows below the order-line units: 5521
rows flagged in either way: 8338
share of the ledger flagged %: 3.51

           ledger_rows  flagged_rows  flagged_share_percent
branch_id                                                  
AHM001           39911          1366                   3.42
CHN001           44029          1482                   3.37
DEL001           37967          1351                   3.56
HYD001           36300          1329                   3.66
KOL001           41272          1429                   3.46
PUN001           37751          1381                   3.66

p-value, flagged against branch: 0.1089


In [8]:
late_value = paid.groupby('branch_id')['is_late'].mean() * 100
cover_value = risk_stock.groupby('branch_id')['order_line_stock'].sum() / risk_stock.groupby('branch_id')['units_sold'].sum()
flag_value = flag_by_branch['flagged_rows'] / flag_by_branch['ledger_rows'] * 100
loss_value = loss_by_branch['subtracted_units'] / loss_by_branch['handled_units'] * 100

overall = {
    'late': paid['is_late'].mean() * 100,
    'cover': risk_stock['order_line_stock'].sum() / risk_stock['units_sold'].sum(),
    'flagged': flagged_rows['flagged_any'].sum() / len(led) * 100,
    'loss': subtracted_units.sum() / handled_units.sum() * 100}

score_table = pd.DataFrame({
    'late_rate': late_value, 'years_of_cover': cover_value,
    'flagged_share': flag_value, 'loss_rate': loss_value})

score_table['late_index'] = score_table['late_rate'] / overall['late'] * 100
score_table['cover_index'] = score_table['years_of_cover'] / overall['cover'] * 100
score_table['flagged_index'] = score_table['flagged_share'] / overall['flagged'] * 100
score_table['loss_index'] = score_table['loss_rate'] / overall['loss'] * 100

index_columns = ['late_index', 'cover_index', 'flagged_index', 'loss_index']
score_table['risk_score'] = score_table[index_columns].mean(axis=1)
score_table['rank_1_is_riskiest'] = score_table['risk_score'].rank(ascending=False).astype(int)

print('overall values:', {key: round(value, 4) for key, value in overall.items()})
print()
print(score_table[index_columns + ['risk_score', 'rank_1_is_riskiest']].round(1).sort_values('rank_1_is_riskiest'))

overall values: {'late': np.float64(41.5018), 'cover': np.float64(107.3756), 'flagged': np.float64(3.5147), 'loss': np.float64(0.1555)}

           late_index  cover_index  flagged_index  loss_index  risk_score  \
branch_id                                                                   
HYD001           92.0        120.5          104.2       103.9       105.1   
PUN001          104.0        109.5          104.1        88.6       101.5   
CHN001          117.1         82.8           95.8       104.2       100.0   
DEL001           90.9        106.1          101.2        96.4        98.7   
KOL001           95.7         87.6           98.5       112.7        98.6   
AHM001           95.4        103.1           97.4        94.3        97.5   

           rank_1_is_riskiest  
branch_id                      
HYD001                      1  
PUN001                      2  
CHN001                      3  
DEL001                      4  
KOL001                      5  
AHM001                

In [9]:
import numpy as np

index_values = score_table[index_columns]

drop_results = {}
for column in index_columns:
    kept = [name for name in index_columns if name != column]
    drop_results['without ' + column] = index_values[kept].mean(axis=1).rank(ascending=False).astype(int)
drop_table = pd.DataFrame(drop_results)
print('rank when one signal is left out (1 is riskiest):')
print(drop_table)

random_generator = np.random.default_rng(2026)
weight_sets = random_generator.dirichlet(np.ones(4), size=10000)
random_scores = index_values.values @ weight_sets.T
random_ranks = (-random_scores).argsort(axis=0).argsort(axis=0) + 1

weight_summary = pd.DataFrame({
    'rank_1_in_percent_of_sets': (random_ranks == 1).mean(axis=1) * 100,
    'best_rank': random_ranks.min(axis=1),
    'worst_rank': random_ranks.max(axis=1)},
    index=list(index_values.index)).round(1)
print()
print('10,000 random weight sets:')
print(weight_summary)

rank when one signal is left out (1 is riskiest):
           without late_index  without cover_index  without flagged_index  \
branch_id                                                                   
AHM001                      5                    6                      6   
CHN001                      6                    1                      2   
DEL001                      2                    5                      5   
HYD001                      1                    3                      1   
KOL001                      4                    2                      4   
PUN001                      3                    4                      3   

           without loss_index  
branch_id                      
AHM001                      4  
CHN001                      5  
DEL001                      3  
HYD001                      2  
KOL001                      6  
PUN001                      1  

10,000 random weight sets:
        rank_1_in_percent_of_sets  best_rank  wor

In [10]:
columns_used = {
    'invoices': (invoices, ['invoice_id', 'branch_id', 'due_date'], []),
    'payments': (payments, ['payment_id', 'invoice_id', 'payment_date'], []),
    'stock ledger': (ledger, ['movement_id', 'product_id', 'branch_id', 'movement_type',
                              'movement_date', 'reference_id'], ['quantity', 'running_balance']),
    'inventory master': (inv, ['product_id', 'branch_id'], ['opening_stock']),
    'purchase orders header': (po_h, ['po_id', 'branch_id', 'po_status'], []),
    'purchase order lines': (po_l, ['po_id', 'product_id'], ['quantity']),
    'sales orders header': (so_h, ['so_id', 'branch_id', 'order_status', 'delivery_date'], []),
    'sales order lines': (so_l, ['so_id', 'product_id'], ['quantity'])}

for table_name, (table, text_columns, number_columns) in columns_used.items():
    print(table_name, '-', len(table), 'rows')
    for column in text_columns + number_columns:
        print('   ', column, '- blank:', int(table[column].isna().sum()))
    for column in number_columns:
        print('   ', column, '- zero:', int((table[column] == 0).sum()),
              '- negative:', int((table[column] < 0).sum()))

invoices - 18033 rows
    invoice_id - blank: 0
    branch_id - blank: 0
    due_date - blank: 0
payments - 19257 rows
    payment_id - blank: 0
    invoice_id - blank: 0
    payment_date - blank: 0
stock ledger - 237230 rows
    movement_id - blank: 0
    product_id - blank: 0
    branch_id - blank: 0
    movement_type - blank: 0
    movement_date - blank: 0
    reference_id - blank: 0
    quantity - blank: 0
    running_balance - blank: 0
    quantity - zero: 0 - negative: 0
    running_balance - zero: 0 - negative: 0
inventory master - 180 rows
    product_id - blank: 0
    branch_id - blank: 0
    opening_stock - blank: 0
    opening_stock - zero: 0 - negative: 0
purchase orders header - 24000 rows
    po_id - blank: 0
    branch_id - blank: 0
    po_status - blank: 0
purchase order lines - 155495 rows
    po_id - blank: 0
    product_id - blank: 0
    quantity - blank: 0
    quantity - zero: 0 - negative: 0
sales orders header - 20000 rows
    so_id - blank: 0
    branch_id - blan

In [11]:
branch_list = set(branches['branch_id'])
print('branches in the branch table:', len(branch_list))
print()

sources = {
    'invoices': set(invoices['branch_id']),
    'joined payments': set(paid['branch_id']),
    'stock ledger': set(ledger['branch_id']),
    'inventory pairs': set(risk_stock.reset_index()['branch_id']),
    'purchase orders': set(po_h['branch_id']),
    'sales orders': set(so_h['branch_id'])}
for source_name, found in sources.items():
    print(source_name, '- branches found:', len(found), '- same as the branch table:', found == branch_list)

print()
print('smallest payments in a branch:', int(late_by_branch['payments'].min()))
print('smallest units sold in a branch:', int(cover_by_branch['units_sold'].min()))
print('smallest units handled in a branch:', int(handled_units.min()))
print('smallest ledger rows in a branch:', int(flag_by_branch['ledger_rows'].min()))
print('pairs with zero units sold in the window:', int((risk_stock['units_sold'] == 0).sum()))

print()
invoice_branch = invoices.merge(so_h[['so_id', 'branch_id']], on='so_id', how='left', suffixes=('', '_order'))
print('invoices with no sales order:', int(invoice_branch['branch_id_order'].isna().sum()))
print('invoices whose branch differs from the order branch:',
      int((invoice_branch['branch_id'] != invoice_branch['branch_id_order']).sum()))
print('clean payments with no clean invoice:', len(clean_payments) - len(paid))

print()
print('purchase order statuses:', po_h['po_status'].value_counts().to_dict())
print('sales order statuses:', so_h['order_status'].value_counts().to_dict())

branches in the branch table: 6

invoices - branches found: 6 - same as the branch table: True
joined payments - branches found: 6 - same as the branch table: True
stock ledger - branches found: 6 - same as the branch table: True
inventory pairs - branches found: 6 - same as the branch table: True
purchase orders - branches found: 6 - same as the branch table: True
sales orders - branches found: 6 - same as the branch table: True

smallest payments in a branch: 2576
smallest units sold in a branch: 27484
smallest units handled in a branch: 3538563
smallest ledger rows in a branch: 36300
pairs with zero units sold in the window: 0

invoices with no sales order: 0
invoices whose branch differs from the order branch: 0
clean payments with no clean invoice: 409

purchase order statuses: {'Received': 21630, 'Cancelled': 2370}
sales order statuses: {'Delivered': 18033, 'Cancelled': 1967}


In [12]:
print('branch ids in the branch table:', len(branches), '- distinct:', branches['branch_id'].nunique())
print('ledger rows:', len(ledger), '- distinct movement ids:', ledger['movement_id'].nunique())
print('inventory rows:', len(inv), '- distinct product and branch pairs:',
      len(inv.drop_duplicates(['product_id', 'branch_id'])))
print('purchase orders:', len(po_h), '- distinct ids:', po_h['po_id'].nunique())
print('sales orders:', len(so_h), '- distinct ids:', so_h['so_id'].nunique())

print()
print('invoice rows in repeated id groups:', int((invoices['invoice_id'].map(invoice_counts) > 1).sum()))
print('payment rows in repeated id groups:', int((payments['payment_id'].map(payment_counts) > 1).sum()))
print('invoice ids still repeated after leaving them out:', int(clean_invoices['invoice_id'].duplicated().sum()))
print('payment ids still repeated after leaving them out:', int(clean_payments['payment_id'].duplicated().sum()))
print('payment rows identical to another payment row after leaving them out:', int(clean_payments.duplicated().sum()))

branch ids in the branch table: 6 - distinct: 6
ledger rows: 237230 - distinct movement ids: 237230
inventory rows: 180 - distinct product and branch pairs: 180
purchase orders: 24000 - distinct ids: 24000
sales orders: 20000 - distinct ids: 20000

invoice rows in repeated id groups: 393
payment rows in repeated id groups: 403
invoice ids still repeated after leaving them out: 0
payment ids still repeated after leaving them out: 0
payment rows identical to another payment row after leaving them out: 0


In [13]:
print('payments added up by branch:', int(late_by_branch['payments'].sum()),
      '- late:', int(late_by_branch['late'].sum()))
print('late rate overall %:', round(overall['late'], 2))
print()
print('opening stock total:', int(risk_stock['opening_stock'].sum()))
print('received total:', int(risk_stock['received'].sum()))
print('delivered total:', int(risk_stock['delivered'].sum()))
print('net adjustments total:', int(risk_stock['net_adjustments'].sum()))
print('opening + received - delivered + net adjustments:',
      int(risk_stock['opening_stock'].sum() + risk_stock['received'].sum()
          - risk_stock['delivered'].sum() + risk_stock['net_adjustments'].sum()))
print('order-line stock added up by branch:', int(cover_by_branch['order_line_stock'].sum()))
print('units sold added up by branch:', int(cover_by_branch['units_sold'].sum()))
print('years of cover overall:', round(overall['cover'], 2))
print()
print('ledger rows added up by branch:', int(flag_by_branch['ledger_rows'].sum()),
      '- flagged:', int(flag_by_branch['flagged_rows'].sum()))
print('flagged share overall %:', round(overall['flagged'], 2))
print()
print('units handled added up by branch:', int(loss_by_branch['handled_units'].sum()),
      '- subtracted:', int(loss_by_branch['subtracted_units'].sum()))
print('loss rate overall %:', round(overall['loss'], 4))
print()
print('each index averaged with its own branch weights (should be 100):')
print('late:', round((score_table['late_index'] * late_by_branch['payments']).sum() / late_by_branch['payments'].sum(), 2))
print('cover:', round((score_table['cover_index'] * cover_by_branch['units_sold']).sum() / cover_by_branch['units_sold'].sum(), 2))
print('flagged:', round((score_table['flagged_index'] * flag_by_branch['ledger_rows']).sum() / flag_by_branch['ledger_rows'].sum(), 2))
print('loss:', round((score_table['loss_index'] * loss_by_branch['handled_units']).sum() / loss_by_branch['handled_units'].sum(), 2))

payments added up by branch: 18445 - late: 7655
late rate overall %: 41.5

opening stock total: 32969
received total: 22387123
delivered total: 1234939
net adjustments total: 62
opening + received - delivered + net adjustments: 21185215
order-line stock added up by branch: 21185215
units sold added up by branch: 197300
years of cover overall: 107.38

ledger rows added up by branch: 237230 - flagged: 8338
flagged share overall %: 3.51

units handled added up by branch: 21522627 - subtracted: 33464
loss rate overall %: 0.1555

each index averaged with its own branch weights (should be 100):
late: 100.0
cover: 100.0
flagged: 100.0
loss: 100.0


In [14]:
from collections import defaultdict

seen_invoice = defaultdict(int)
for row in invoices.itertuples(index=False):
    seen_invoice[row.invoice_id] += 1
seen_payment = defaultdict(int)
for row in payments.itertuples(index=False):
    seen_payment[row.payment_id] += 1
good_invoice = {row.invoice_id: (row.branch_id, row.due_date)
                for row in invoices.itertuples(index=False) if seen_invoice[row.invoice_id] == 1}

loop_payments = defaultdict(int)
loop_late = defaultdict(int)
for row in payments.itertuples(index=False):
    if seen_payment[row.payment_id] == 1 and row.invoice_id in good_invoice:
        branch, due = good_invoice[row.invoice_id]
        loop_payments[branch] += 1
        if row.payment_date > due:
            loop_late[branch] += 1

po_info = {row.po_id: (row.branch_id, row.po_status) for row in po_h.itertuples(index=False)}
so_info = {row.so_id: (row.branch_id, row.order_status, row.delivery_date) for row in so_h.itertuples(index=False)}
opening = {(row.product_id, row.branch_id): row.opening_stock for row in inv.itertuples(index=False)}

loop_received = defaultdict(int)
line_units_by_order_product = defaultdict(int)
for row in po_l.itertuples(index=False):
    branch, status = po_info[row.po_id]
    line_units_by_order_product[(row.po_id, row.product_id)] += row.quantity
    if status == 'Received':
        loop_received[(row.product_id, branch)] += row.quantity

loop_delivered = defaultdict(int)
loop_sold = defaultdict(int)
for row in so_l.itertuples(index=False):
    branch, status, delivery = so_info[row.so_id]
    line_units_by_order_product[(row.so_id, row.product_id)] += row.quantity
    if status == 'Delivered':
        loop_delivered[(row.product_id, branch)] += row.quantity
        if delivery > '2024-01-14':
            loop_sold[(row.product_id, branch)] += row.quantity

ledger_rows_by_pair = defaultdict(list)
ledger_units_by_order_product = defaultdict(int)
for row in ledger.itertuples(index=False):
    ledger_rows_by_pair[(row.product_id, row.branch_id)].append(row)
    if row.movement_type in ('IN', 'OUT'):
        ledger_units_by_order_product[(row.reference_id, row.product_id)] += row.quantity

loop_handled = defaultdict(int)
loop_subtracted = defaultdict(int)
loop_net = defaultdict(int)
for pair_key, rows_of_pair in ledger_rows_by_pair.items():
    balance = opening[pair_key]
    for row in sorted(rows_of_pair, key=lambda item: item.movement_date):
        change = row.running_balance - balance
        balance = row.running_balance
        if row.movement_type in ('IN', 'OUT'):
            loop_handled[pair_key[1]] += row.quantity
        elif row.movement_type == 'ADJUSTMENT':
            loop_net[pair_key] += change
            if change < 0:
                loop_subtracted[pair_key[1]] += row.quantity

loop_ledger_rows = defaultdict(int)
loop_flagged = defaultdict(int)
for row in ledger.itertuples(index=False):
    loop_ledger_rows[row.branch_id] += 1
    if row.movement_type == 'ADJUSTMENT':
        continue
    status = po_info[row.reference_id][1] if row.movement_type == 'IN' else so_info[row.reference_id][1]
    key = (row.reference_id, row.product_id)
    if status == 'Cancelled' or ledger_units_by_order_product[key] < line_units_by_order_product[key]:
        loop_flagged[row.branch_id] += 1

loop_stock = defaultdict(int)
loop_units_sold = defaultdict(int)
for pair_key, start in opening.items():
    loop_stock[pair_key[1]] += start + loop_received[pair_key] - loop_delivered[pair_key] + loop_net[pair_key]
    loop_units_sold[pair_key[1]] += loop_sold[pair_key]

branch_names = sorted(branch_list)
second = pd.DataFrame(index=branch_names)
second['late_rate'] = [loop_late[b] / loop_payments[b] * 100 for b in branch_names]
second['years_of_cover'] = [loop_stock[b] / loop_units_sold[b] for b in branch_names]
second['flagged_share'] = [loop_flagged[b] / loop_ledger_rows[b] * 100 for b in branch_names]
second['loss_rate'] = [loop_subtracted[b] / loop_handled[b] * 100 for b in branch_names]

overall_second = {
    'late': sum(loop_late.values()) / sum(loop_payments.values()) * 100,
    'cover': sum(loop_stock.values()) / sum(loop_units_sold.values()),
    'flagged': sum(loop_flagged.values()) / sum(loop_ledger_rows.values()) * 100,
    'loss': sum(loop_subtracted.values()) / sum(loop_handled.values()) * 100}
second_score = ((second['late_rate'] / overall_second['late'] * 100
                 + second['years_of_cover'] / overall_second['cover'] * 100
                 + second['flagged_share'] / overall_second['flagged'] * 100
                 + second['loss_rate'] / overall_second['loss'] * 100) / 4)

print('totals from the loops:')
print('payments', sum(loop_payments.values()), '- late', sum(loop_late.values()))
print('order-line stock', int(sum(loop_stock.values())), '- units sold', sum(loop_units_sold.values()))
print('ledger rows', sum(loop_ledger_rows.values()), '- flagged', sum(loop_flagged.values()))
print('units handled', sum(loop_handled.values()), '- subtracted', sum(loop_subtracted.values()))
print()
print('largest difference between the two methods:')
print('late rate:', round(float((second['late_rate'] - score_table['late_rate']).abs().max()), 10))
print('years of cover:', round(float((second['years_of_cover'] - score_table['years_of_cover']).abs().max()), 10))
print('flagged share:', round(float((second['flagged_share'] - score_table['flagged_share']).abs().max()), 10))
print('loss rate:', round(float((second['loss_rate'] - score_table['loss_rate']).abs().max()), 10))
print('risk score:', round(float((second_score - score_table['risk_score']).abs().max()), 10))
print('same ranking:', list(second_score.rank(ascending=False).astype(int)) == list(score_table['rank_1_is_riskiest']))

totals from the loops:
payments 18445 - late 7655
order-line stock 21185215 - units sold 197300
ledger rows 237230 - flagged 8338
units handled 21522627 - subtracted 33464

largest difference between the two methods:
late rate: 0.0
years of cover: 0.0
flagged share: 0.0
loss rate: 0.0
risk score: 0.0
same ranking: True


In [15]:
print('range of each index and of the score across the six branches:')
for column in index_columns + ['risk_score']:
    print(column, '-', round(score_table[column].min(), 1), 'to', round(score_table[column].max(), 1))

pair_cover = risk_stock['years_of_cover']
first_quarter = pair_cover.quantile(0.25)
third_quarter = pair_cover.quantile(0.75)
low_limit = first_quarter - 1.5 * (third_quarter - first_quarter)
high_limit = third_quarter + 1.5 * (third_quarter - first_quarter)
outside = risk_stock[(pair_cover < low_limit) | (pair_cover > high_limit)]

print()
print('normal range of years of cover per pair:', round(low_limit, 1), 'to', round(high_limit, 1))
print('pairs outside the range:', len(outside))
print(outside[['order_line_stock', 'units_sold', 'years_of_cover']].round(2))

kept = risk_stock.drop(outside.index)
kept_cover = kept.groupby('branch_id')['order_line_stock'].sum() / kept.groupby('branch_id')['units_sold'].sum()
kept_overall = kept['order_line_stock'].sum() / kept['units_sold'].sum()

trial = score_table[['late_index', 'flagged_index', 'loss_index']].copy()
trial['cover_index'] = kept_cover / kept_overall * 100
trial['risk_score'] = trial.mean(axis=1)
trial['rank_without_outlier_pairs'] = trial['risk_score'].rank(ascending=False).astype(int)
trial['rank_all_pairs'] = score_table['rank_1_is_riskiest']

print()
print('cover index, score and rank without the outlier pairs, next to the original rank:')
print(trial[['cover_index', 'risk_score', 'rank_without_outlier_pairs', 'rank_all_pairs']].round(1))

range of each index and of the score across the six branches:
late_index - 90.9 to 117.1
cover_index - 82.8 to 120.5
flagged_index - 95.8 to 104.2
loss_index - 88.6 to 112.7
risk_score - 97.5 to 105.1

normal range of years of cover per pair: 54.2 to 166.5
pairs outside the range: 1
                      order_line_stock  units_sold  years_of_cover
product_id branch_id                                              
P005       HYD001             122924.0         715          171.92

cover index, score and rank without the outlier pairs, next to the original rank:
           cover_index  risk_score  rank_without_outlier_pairs  rank_all_pairs
branch_id                                                                     
AHM001           103.3        97.6                           6               6
CHN001            83.0       100.0                           3               3
DEL001           106.4        98.7                           4               4
HYD001           119.7       104.9   

In [16]:
paid['due_year'] = paid['due_date'].dt.year

print('first and last due date:', paid['due_date'].min().date(), 'to', paid['due_date'].max().date())
print('first and last payment date:', paid['payment_date'].min().date(), 'to', paid['payment_date'].max().date())
print('first and last ledger date:', led['movement_date'].min().date(), 'to', led['movement_date'].max().date())
window_dates = delivered_lines.loc[delivered_lines['delivery_date'] > '2024-01-14', 'delivery_date']
print('first and last delivery date in the units sold window:', window_dates.min().date(), 'to', window_dates.max().date())

late_by_year = paid.pivot_table(index='due_year', columns='branch_id', values='is_late', aggfunc='mean') * 100
payments_by_year = paid.groupby('due_year').size()

print()
print('late rate % by branch and due year:')
print(late_by_year.round(1))
print()
print('payments in each due year:')
print(payments_by_year)

highest_each_year = late_by_year.idxmax(axis=1)
print()
print('branch with the highest late rate in each due year:')
print(highest_each_year)
print('years in which CHN001 is highest:', int((highest_each_year == 'CHN001').sum()), 'of', len(highest_each_year))
print('smallest number of payments in a branch and due year:', int(paid.groupby(['due_year', 'branch_id']).size().min()))

first and last due date: 2019-01-20 to 2025-03-14
first and last payment date: 2019-01-09 to 2025-03-20
first and last ledger date: 2019-01-01 to 2025-01-28
first and last delivery date in the units sold window: 2024-01-15 to 2025-01-13

late rate % by branch and due year:
branch_id  AHM001  CHN001  DEL001  HYD001  KOL001  PUN001
due_year                                                 
2019         41.8    48.1    39.6    39.7    42.7    43.9
2020         40.3    48.2    42.5    39.6    36.9    42.4
2021         40.7    49.3    36.5    35.2    37.6    39.7
2022         40.6    48.7    38.7    36.6    44.0    46.1
2023         37.2    49.0    35.1    41.4    38.2    45.2
2024         38.6    49.1    36.7    38.1    41.9    42.6
2025         31.7    40.7    20.0    28.8    27.8    34.5

payments in each due year:
due_year
2019    2717
2020    3060
2021    3089
2022    3102
2023    3101
2024    3005
2025     371
dtype: int64

branch with the highest late rate in each due year:
due_year
2

In [17]:
import csv
from collections import Counter

def read_rows(file_name):
    with open(BASE_PATH + file_name, newline='') as handle:
        return list(csv.DictReader(handle))

raw_invoices = read_rows('invoices.csv')
raw_payments = read_rows('payments.csv')
raw_ledger_rows = read_rows('stock_ledger.csv')
raw_inventory = read_rows('inventory_master.csv')
raw_po_header = read_rows('purchase_orders_header.csv')
raw_po_lines = read_rows('purchase_orders_lines.csv')
raw_so_header = read_rows('sales_orders_header.csv')
raw_so_lines = read_rows('sales_orders_lines.csv')

three_branches = ['HYD001', 'CHN001', 'AHM001']
by_hand = {b: dict.fromkeys(['payments', 'late', 'stock', 'sold', 'ledger_rows',
                             'flagged', 'handled', 'subtracted'], 0) for b in three_branches}

invoice_times = Counter(r['invoice_id'] for r in raw_invoices)
payment_times = Counter(r['payment_id'] for r in raw_payments)
invoice_lookup = {r['invoice_id']: r for r in raw_invoices if invoice_times[r['invoice_id']] == 1}
for r in raw_payments:
    if payment_times[r['payment_id']] == 1 and r['invoice_id'] in invoice_lookup:
        invoice_row = invoice_lookup[r['invoice_id']]
        branch = invoice_row['branch_id']
        if branch in by_hand:
            by_hand[branch]['payments'] += 1
            if r['payment_date'] > invoice_row['due_date']:
                by_hand[branch]['late'] += 1

po_lookup = {r['po_id']: (r['branch_id'], r['po_status']) for r in raw_po_header}
so_lookup = {r['so_id']: (r['branch_id'], r['order_status'], r['delivery_date']) for r in raw_so_header}
opening_hand = {(r['product_id'], r['branch_id']): int(r['opening_stock']) for r in raw_inventory}

received_hand = Counter()
delivered_hand = Counter()
sold_hand = Counter()
line_units_hand = Counter()
for r in raw_po_lines:
    branch, status = po_lookup[r['po_id']]
    line_units_hand[(r['po_id'], r['product_id'])] += int(r['quantity'])
    if status == 'Received':
        received_hand[(r['product_id'], branch)] += int(r['quantity'])
for r in raw_so_lines:
    branch, status, delivery = so_lookup[r['so_id']]
    line_units_hand[(r['so_id'], r['product_id'])] += int(r['quantity'])
    if status == 'Delivered':
        delivered_hand[(r['product_id'], branch)] += int(r['quantity'])
        if delivery > '2024-01-14':
            sold_hand[(r['product_id'], branch)] += int(r['quantity'])

ledger_units_hand = Counter()
rows_by_pair = {}
for r in raw_ledger_rows:
    if r['movement_type'] in ('IN', 'OUT'):
        ledger_units_hand[(r['reference_id'], r['product_id'])] += int(r['quantity'])
    if r['branch_id'] in by_hand:
        rows_by_pair.setdefault((r['product_id'], r['branch_id']), []).append(r)

for r in raw_ledger_rows:
    if r['branch_id'] not in by_hand:
        continue
    by_hand[r['branch_id']]['ledger_rows'] += 1
    if r['movement_type'] in ('IN', 'OUT'):
        status = po_lookup[r['reference_id']][1] if r['movement_type'] == 'IN' else so_lookup[r['reference_id']][1]
        key = (r['reference_id'], r['product_id'])
        if status == 'Cancelled' or ledger_units_hand[key] < line_units_hand[key]:
            by_hand[r['branch_id']]['flagged'] += 1

net_hand = Counter()
for pair_key, rows_of_pair in rows_by_pair.items():
    balance = opening_hand[pair_key]
    for r in sorted(rows_of_pair, key=lambda item: item['movement_date']):
        new_balance = int(r['running_balance'])
        change = new_balance - balance
        balance = new_balance
        if r['movement_type'] in ('IN', 'OUT'):
            by_hand[pair_key[1]]['handled'] += int(r['quantity'])
        elif r['movement_type'] == 'ADJUSTMENT':
            net_hand[pair_key] += change
            if change < 0:
                by_hand[pair_key[1]]['subtracted'] += int(r['quantity'])

for pair_key, start in opening_hand.items():
    if pair_key[1] in by_hand:
        by_hand[pair_key[1]]['stock'] += (start + received_hand[pair_key] - delivered_hand[pair_key]
                                          + net_hand[pair_key])
        by_hand[pair_key[1]]['sold'] += sold_hand[pair_key]

table_numbers = {
    'payments': late_by_branch['payments'], 'late': late_by_branch['late'],
    'stock': cover_by_branch['order_line_stock'], 'sold': cover_by_branch['units_sold'],
    'ledger_rows': flag_by_branch['ledger_rows'], 'flagged': flag_by_branch['flagged_rows'],
    'handled': loss_by_branch['handled_units'], 'subtracted': loss_by_branch['subtracted_units']}

for b in three_branches:
    print(b)
    for name, series in table_numbers.items():
        print('   ', name, '- by hand:', int(by_hand[b][name]), '- table:', int(series[b]),
              '- agree:', int(by_hand[b][name]) == int(series[b]))

print()
for b in three_branches:
    h = by_hand[b]
    late_index = h['late'] / h['payments'] * 100 / overall['late'] * 100
    cover_index = h['stock'] / h['sold'] / overall['cover'] * 100
    flagged_index = h['flagged'] / h['ledger_rows'] * 100 / overall['flagged'] * 100
    loss_index = h['subtracted'] / h['handled'] * 100 / overall['loss'] * 100
    print(b, '- score by hand:', round((late_index + cover_index + flagged_index + loss_index) / 4, 2),
          '- table:', round(score_table.loc[b, 'risk_score'], 2))

HYD001
    payments - by hand: 2576 - table: 2576 - agree: True
    late - by hand: 984 - table: 984 - agree: True
    stock - by hand: 3555196 - table: 3555196 - agree: True
    sold - by hand: 27484 - table: 27484 - agree: True
    ledger_rows - by hand: 36300 - table: 36300 - agree: True
    flagged - by hand: 1329 - table: 1329 - agree: True
    handled - by hand: 3548234 - table: 3548234 - agree: True
    subtracted - by hand: 5730 - table: 5730 - agree: True
CHN001
    payments - by hand: 3732 - table: 3732 - agree: True
    late - by hand: 1814 - table: 1814 - agree: True
    stock - by hand: 3548773 - table: 3548773 - agree: True
    sold - by hand: 39930 - table: 39930 - agree: True
    ledger_rows - by hand: 44029 - table: 44029 - agree: True
    flagged - by hand: 1482 - table: 1482 - agree: True
    handled - by hand: 3692804 - table: 3692804 - agree: True
    subtracted - by hand: 5980 - table: 5980 - agree: True
AHM001
    payments - by hand: 3196 - table: 3196 - agree: T

In [18]:
average_stock = cover_by_branch['order_line_stock'].mean()
average_sold = cover_by_branch['units_sold'].mean()

decomposition = pd.DataFrame({
    'actual_cover': cover_by_branch['order_line_stock'] / cover_by_branch['units_sold'],
    'cover_if_equal_stock': average_stock / cover_by_branch['units_sold'],
    'cover_if_equal_sales': cover_by_branch['order_line_stock'] / average_sold})

print(decomposition.round(2))
print()
for column in decomposition.columns:
    print(column, '- highest over lowest:', round(decomposition[column].max() / decomposition[column].min(), 3))

           actual_cover  cover_if_equal_stock  cover_if_equal_sales
branch_id                                                          
AHM001           110.66                112.53                105.59
CHN001            88.87                 88.43                107.92
DEL001           113.98                111.86                109.41
HYD001           129.36                128.47                108.12
KOL001            94.02                 96.62                104.48
PUN001           117.61                116.14                108.74

actual_cover - highest over lowest: 1.455
cover_if_equal_stock - highest over lowest: 1.453
cover_if_equal_sales - highest over lowest: 1.047


In [19]:
signals = score_table[['late_rate', 'years_of_cover', 'flagged_share', 'loss_rate']]
min_max = (signals - signals.min()) / (signals.max() - signals.min()) * 100
min_max_score = min_max.mean(axis=1)
two_signal_score = score_table[['late_index', 'cover_index']].mean(axis=1)

compare = pd.DataFrame({
    'headline_score': score_table['risk_score'],
    'headline_rank': score_table['rank_1_is_riskiest'],
    'min_max_score': min_max_score,
    'min_max_rank': min_max_score.rank(ascending=False).astype(int),
    'two_signal_score': two_signal_score,
    'two_signal_rank': two_signal_score.rank(ascending=False).astype(int)})
print(compare.round(1))

overall_values = pd.Series({'late_rate': overall['late'], 'years_of_cover': overall['cover'],
                            'flagged_share': overall['flagged'], 'loss_rate': overall['loss']})
spread = (signals.max() - signals.min()) / overall_values * 100
print()
print('highest minus lowest branch value, as a share of the overall figure (%):')
print(spread.round(1))

           headline_score  headline_rank  min_max_score  min_max_rank  \
branch_id                                                               
AHM001               97.5              6           28.5             6   
CHN001              100.0              3           41.1             3   
DEL001               98.7              4           39.9             5   
HYD001              105.1              1           66.9             1   
KOL001               98.6              5           41.0             4   
PUN001              101.5              2           55.0             2   

           two_signal_score  two_signal_rank  
branch_id                                     
AHM001                 99.3                4  
CHN001                 99.9                3  
DEL001                 98.5                5  
HYD001                106.3                2  
KOL001                 91.7                6  
PUN001                106.8                1  

highest minus lowest branch value, as 